
# Supervised Regime Classifier


## 1. Imports & Constants

In [ ]:
from pathlib import Path
import warnings
from functools import reduce
from typing import Optional, List, Dict, Tuple, Any

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import ttest_ind, f_oneway, kruskal

warnings.filterwarnings("ignore")

try:
    from lightgbm import LGBMClassifier
    _HAS_LGBM = True
except ImportError:
    _HAS_LGBM = False
    print("LightGBM not found; will use fallback classifier.")

INDEX_PATH = "index_data.csv"
RETURN_PATH = "return.csv"
OUTPUT_PATH = "supervised_full_sample_predictions.csv"

MAIN_CODES = ["000001", "399001", "000020", "000905", "399005", "000852"]
GROWTH_CODES = ["000688", "399006", "399673"]
RISK_CODES = ["399006", "000688", "000852"]
TRAIN_END = 20241231
HOLDOUT_START = 20250101
HOLDOUT_END = 20251231
RECENT_PRIOR_START = 20260501
RECENT_PRIOR_END = 20260528
ANN_FACTOR = 242
EPS = 1e-12


## 2. Helper Functions

In [ ]:

def rolling_zscore(series: pd.Series, window: int = 252, min_periods: int = 60) -> pd.Series:
    mu = series.rolling(window, min_periods=min_periods).mean()
    sd = series.rolling(window, min_periods=min_periods).std()
    return (series - mu) / sd.replace(0, np.nan)


def calc_max_drawdown(ret_series: pd.Series) -> float:
    if len(ret_series) == 0:
        return np.nan
    nav = (1 + ret_series.fillna(0)).cumprod()
    peak = nav.cummax()
    return (nav / peak - 1).min()


def calc_tstat(x) -> float:
    x = pd.Series(x).dropna()
    if len(x) < 2:
        return np.nan
    s = x.std()
    if pd.isna(s) or s < EPS:
        return np.nan
    return float(x.mean() / (s / np.sqrt(len(x))))


def summarize_strategy_by_state(
    df: pd.DataFrame,
    state_col: str = "state",
    ret_col: str = "ret",
    ann_factor: int = ANN_FACTOR
) -> pd.DataFrame:
    rows = []
    total = len(df[ret_col].dropna()) if ret_col in df.columns else 0
    for s, g in df.groupby(state_col):
        r = g[ret_col].dropna() if ret_col in g.columns else pd.Series(dtype=float)
        if len(r) == 0:
            continue
        mean_r = r.mean()
        std_r = r.std()
        rows.append({
            "state": s,
            "count": len(r),
            "count_pct": len(r) / total if total else np.nan,
            "mean_ret": mean_r,
            "std_ret": std_r,
            "tstat": calc_tstat(r),
            "sharpe_ann": mean_r / std_r * np.sqrt(ann_factor) if std_r > EPS else np.nan,
            "win_rate": (r > 0).mean(),
            "max_drawdown": calc_max_drawdown(r)
        })
    return pd.DataFrame(rows).sort_values("state").reset_index(drop=True)


def state_return_hypothesis_tests(
    df: pd.DataFrame,
    state_col: str = "state",
    ret_col: str = "ret"
) -> Dict[str, Any]:
    work = df[[state_col, ret_col]].dropna().copy()
    states = sorted(work[state_col].unique().tolist())
    groups = [work.loc[work[state_col] == s, ret_col].dropna() for s in states]
    groups = [g for g in groups if len(g) > 1]

    result: Dict[str, Any] = {}
    if len(groups) >= 2:
        try:
            f, p = f_oneway(*groups)
            result["anova_F"] = f
            result["anova_p"] = p
        except Exception:
            result["anova_F"] = np.nan
            result["anova_p"] = np.nan
        try:
            _, p_kw = kruskal(*groups)
            result["kruskal_p"] = p_kw
        except Exception:
            result["kruskal_p"] = np.nan
    else:
        result["anova_F"] = np.nan
        result["anova_p"] = np.nan
        result["kruskal_p"] = np.nan

    pair_rows = []
    for i in range(len(states)):
        for j in range(i + 1, len(states)):
            s1, s2 = states[i], states[j]
            x1 = work.loc[work[state_col] == s1, ret_col].dropna()
            x2 = work.loc[work[state_col] == s2, ret_col].dropna()
            if len(x1) < 2 or len(x2) < 2:
                continue
            t, p = ttest_ind(x1, x2, equal_var=False, nan_policy="omit")
            pair_rows.append({
                "state_i": s1,
                "state_j": s2,
                "mean_i": x1.mean(),
                "mean_j": x2.mean(),
                "mean_diff": x1.mean() - x2.mean(),
                "t_stat": t,
                "p_value": p,
            })
    result["pairwise_tests"] = pd.DataFrame(pair_rows)
    return result


def evaluate_oos_states(oos_df: pd.DataFrame, label: str = "OOS") -> Dict[str, Any]:
    if len(oos_df) == 0:
        print(f"[{label}] no OOS rows")
        return {}

    print(f"[{label}] state counts")
    print(oos_df["state"].value_counts().sort_index())
    print()

    perf = summarize_strategy_by_state(oos_df, state_col="state", ret_col="ret")
    tests = state_return_hypothesis_tests(oos_df, state_col="state", ret_col="ret")
    print(f"[{label}] performance by state")
    display(perf.round(6))
    print(f"[{label}] tests")
    print({
        "anova_p": None if pd.isna(tests["anova_p"]) else round(float(tests["anova_p"]), 6),
        "kruskal_p": None if pd.isna(tests["kruskal_p"]) else round(float(tests["kruskal_p"]), 6),
    })
    if len(tests["pairwise_tests"]) > 0:
        display(tests["pairwise_tests"].round(6))
    return {"performance_by_state": perf, "tests": tests}


## 3. Feature Engineering

In [ ]:
def build_single_index_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values("trade_date").reset_index(drop=True)

    df["r1"] = df["close"] / df["preclose"].replace(0, np.nan) - 1
    for w in [3, 5, 10, 20, 60]:
        df[f"r{w}"] = df["close"] / df["close"].shift(w).replace(0, np.nan) - 1

    df["amp"] = df["high"] / df["low"].replace(0, np.nan) - 1
    df["intraday_ret"] = df["close"] / df["open"].replace(0, np.nan) - 1
    df["gap"] = df["open"] / df["preclose"].replace(0, np.nan) - 1

    for w in [5, 20, 60]:
        df[f"vol_{w}"] = df["r1"].rolling(w, min_periods=max(3, w // 3)).std()

    for w in [20, 60]:
        df[f"ma{w}"] = df["close"].rolling(w, min_periods=max(5, w // 3)).mean()
        df[f"ma{w}_bias"] = df["close"] / df[f"ma{w}"].replace(0, np.nan) - 1

    df["ma20_over_ma60"] = df["ma20"] / df["ma60"].replace(0, np.nan) - 1

    for w in [20, 60]:
        df[f"liq_z_{w}"] = rolling_zscore(df["volume"], w, max(10, w // 3))

    df["up_frac_10"] = df["r1"].gt(0).rolling(10, min_periods=5).mean()
    df["up_frac_20"] = df["r1"].gt(0).rolling(20, min_periods=10).mean()
    df["dist_from_high_20"] = df["close"] / df["high"].rolling(20, min_periods=10).max().replace(0, np.nan) - 1
    df["vol_shock"] = df["vol_5"] / df["vol_20"].replace(0, np.nan) - 1

    return df


def build_market_feature_table(df_raw: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    for code, g in df_raw.groupby("idx"):
        x = build_single_index_features(g)
        x["idx"] = str(code % 1000000).zfill(6)
        pieces.append(x)

    feat_long = pd.concat(pieces, axis=0, ignore_index=True)
    keep_cols = [
        "trade_date", "idx",
        "r1", "r5", "r20", "r60",
        "vol_5", "vol_20", "vol_60",
        "ma20_bias", "ma60_bias", "ma20_over_ma60",
        "liq_z_20", "up_frac_10", "up_frac_20",
        "dist_from_high_20", "vol_shock",
    ]
    feat_long = feat_long[keep_cols].copy()

    wide = []
    for idx_name, g in feat_long.groupby("idx"):
        gg = g.drop(columns=["idx"]).copy()
        rename_map = {c: f"{idx_name}__{c}" for c in gg.columns if c != "trade_date"}
        wide.append(gg.rename(columns=rename_map))

    feat = reduce(lambda left, right: pd.merge(left, right, on="trade_date", how="outer"), wide)
    feat = feat.sort_values("trade_date").reset_index(drop=True)

    def existing_cols(codes: List[str], suffix: str) -> List[str]:
        return [f"{code}__{suffix}" for code in codes if f"{code}__{suffix}" in feat.columns]

    for suffix in ["r1", "r5", "r20", "r60", "vol_5", "vol_20", "vol_60", "ma20_bias", "ma60_bias", "up_frac_10", "up_frac_20"]:
        main_cols = existing_cols(MAIN_CODES, suffix)
        growth_cols = existing_cols(GROWTH_CODES, suffix)
        feat[f"main_{suffix}"] = feat[main_cols].mean(axis=1) if len(main_cols) else np.nan
        feat[f"growth_{suffix}"] = feat[growth_cols].mean(axis=1) if len(growth_cols) else np.nan
        feat[f"style_spread_{suffix}"] = feat[f"main_{suffix}"] - feat[f"growth_{suffix}"]

    for suffix in ["r1", "r5", "r20", "r60"]:
        risk_cols = existing_cols(RISK_CODES, suffix)
        base_col = f"000001__{suffix}" if f"000001__{suffix}" in feat.columns else None
        feat[f"risk_on_{suffix}"] = feat[risk_cols].mean(axis=1) - feat[base_col] if base_col and len(risk_cols) else np.nan

    all_r1_cols = [c for c in feat.columns if c.endswith("__r1")]
    all_r5_cols = [c for c in feat.columns if c.endswith("__r5")]
    all_ma20_bias_cols = [c for c in feat.columns if c.endswith("__ma20_bias")]

    feat["breadth_up_ratio_1"] = feat[all_r1_cols].gt(0).mean(axis=1)
    feat["breadth_up_ratio_5"] = feat[all_r5_cols].gt(0).mean(axis=1)
    feat["breadth_trend_ratio"] = feat[all_ma20_bias_cols].gt(0).mean(axis=1)
    feat["cross_dispersion_r1"] = feat[all_r1_cols].std(axis=1)
    feat["cross_dispersion_r5"] = feat[all_r5_cols].std(axis=1)

    feat["main_vol_shock"] = feat["main_vol_5"] / feat["main_vol_20"].replace(0, np.nan) - 1
    feat["vol_spread_vol_5"] = feat["main_vol_5"] - feat["growth_vol_5"]
    feat["vol_spread_vol_20"] = feat["main_vol_20"] - feat["growth_vol_20"]
    feat["vol_spread_vol_60"] = feat["main_vol_60"] - feat["growth_vol_60"]

    for code in ["399006", "000688", "000905", "000852", "399001"]:
        for suffix in ["r1", "r5", "r20"]:
            left = f"{code}__{suffix}"
            right = f"000001__{suffix}"
            if left in feat.columns and right in feat.columns:
                feat[f"{code}_minus_000001_{suffix}"] = feat[left] - feat[right]

    feat["growth_lead_r5"] = feat["growth_r5"] - feat["main_r5"]
    feat["growth_lead_r20"] = feat["growth_r20"] - feat["main_r20"]
    feat["growth_lead_r60"] = feat["growth_r60"] - feat["main_r60"]
    feat["growth_strength_gap"] = feat["growth_ma20_bias"] - feat["main_ma20_bias"]
    feat["growth_breadth_gap"] = feat["growth_up_frac_20"] - feat["main_up_frac_20"]
    feat["siphon_headwind"] = feat["growth_lead_r20"].clip(lower=0) * (1 - feat["main_up_frac_20"].clip(0, 1))
    feat["rotation_stress"] = feat["growth_lead_r5"].clip(lower=0) * feat["cross_dispersion_r1"]
    feat["growth_vol_headwind"] = feat["growth_lead_r20"].clip(lower=0) * (feat["growth_vol_20"] - feat["main_vol_20"])
    feat["headwind_score_raw"] = feat[["siphon_headwind", "rotation_stress", "growth_vol_headwind"]].sum(axis=1, min_count=1)

    for col in [
        "style_spread_r1", "style_spread_r5", "style_spread_r60",
        "risk_on_r1", "risk_on_r5",
        "breadth_up_ratio_1", "breadth_up_ratio_5", "cross_dispersion_r1",
        "main_ma20_bias", "main_ma60_bias", "growth_ma20_bias", "growth_ma60_bias",
        "growth_lead_r5", "growth_lead_r20", "growth_strength_gap",
        "siphon_headwind", "rotation_stress", "headwind_score_raw",
    ]:
        if col in feat.columns:
            feat[f"{col}_chg5"] = feat[col].diff(5)

    return feat.sort_values("trade_date").reset_index(drop=True)


def select_model_features(feat: pd.DataFrame) -> List[str]:
    preferred = [
        "000001__r1", "000001__r5", "000001__r20", "000001__r60", "000001__vol_5", "000001__vol_20", "000001__vol_60",
        "000001__ma20_bias", "000001__ma60_bias", "000001__liq_z_20", "000001__ma20_over_ma60",
        "000001__up_frac_10", "000001__up_frac_20", "000001__dist_from_high_20",
        "399001__r1", "399001__r5", "399001__r20", "399001__r60", "399001__vol_5", "399001__vol_20", "399001__vol_60", "399001__ma20_bias", "399001__ma60_bias", "399001__liq_z_20",
        "000905__r1", "000905__r5", "000905__r20", "000905__r60", "000905__vol_5", "000905__vol_20", "000905__vol_60", "000905__ma20_bias", "000905__ma60_bias", "000905__liq_z_20",
        "000852__r1", "000852__r5", "000852__r20", "000852__r60", "000852__vol_5", "000852__vol_20", "000852__vol_60", "000852__ma20_bias", "000852__ma60_bias", "000852__liq_z_20",
        "399006__r1", "399006__r5", "399006__r20", "399006__r60", "399006__vol_5", "399006__vol_20", "399006__vol_60", "399006__ma20_bias", "399006__ma60_bias", "399006__liq_z_20",
        "000688__r1", "000688__r5", "000688__r20", "000688__r60", "000688__vol_5", "000688__vol_20", "000688__vol_60", "000688__ma20_bias", "000688__ma60_bias", "000688__liq_z_20",
        "main_r1", "main_r5", "main_r20", "main_r60", "growth_r1", "growth_r5", "growth_r20", "growth_r60",
        "style_spread_r1", "style_spread_r5", "style_spread_r20", "style_spread_r60", "style_spread_ma20_bias", "style_spread_ma60_bias",
        "risk_on_r1", "risk_on_r5", "risk_on_r20", "risk_on_r60", "breadth_up_ratio_1", "breadth_up_ratio_5",
        "breadth_trend_ratio", "cross_dispersion_r1", "cross_dispersion_r5", "main_ma20_bias", "main_ma60_bias",
        "growth_ma20_bias", "growth_ma60_bias", "main_up_frac_10", "main_up_frac_20", "main_vol_shock",
        "vol_spread_vol_5", "vol_spread_vol_20", "vol_spread_vol_60",
        "399006_minus_000001_r1", "399006_minus_000001_r5", "399006_minus_000001_r20",
        "000688_minus_000001_r1", "000688_minus_000001_r5", "000688_minus_000001_r20",
        "000905_minus_000001_r1", "000905_minus_000001_r5", "000905_minus_000001_r20",
        "000852_minus_000001_r1", "000852_minus_000001_r5", "000852_minus_000001_r20",
        "style_spread_r1_chg5", "style_spread_r5_chg5", "style_spread_r60_chg5",
        "risk_on_r1_chg5", "risk_on_r5_chg5", "breadth_up_ratio_1_chg5", "breadth_up_ratio_5_chg5",
        "cross_dispersion_r1_chg5", "main_ma20_bias_chg5", "main_ma60_bias_chg5", "growth_ma20_bias_chg5", "growth_ma60_bias_chg5",
        "growth_lead_r5", "growth_lead_r20", "growth_lead_r60", "growth_strength_gap", "growth_breadth_gap",
        "siphon_headwind", "rotation_stress", "growth_vol_headwind", "headwind_score_raw",
        "growth_lead_r5_chg5", "growth_lead_r20_chg5", "growth_strength_gap_chg5",
        "siphon_headwind_chg5", "rotation_stress_chg5", "headwind_score_raw_chg5",
    ]
    return [c for c in preferred if c in feat.columns]


## 4. Supervised Learning Pipeline

In [ ]:
def build_return_table(ret_df: pd.DataFrame) -> pd.DataFrame:
    ret_use = ret_df.copy()
    ret_use["trade_date"] = ret_use["date"].astype(int)
    ret_use["ret"] = ret_use["0"].astype(float)
    return ret_use[["trade_date", "ret"]].sort_values("trade_date").reset_index(drop=True)


def build_feature_matrix(
    df_raw: pd.DataFrame,
    feature_cols: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, List[str]]:
    feat = build_market_feature_table(df_raw).sort_values("trade_date").reset_index(drop=True)
    if feature_cols is None:
        feature_cols = select_model_features(feat)
    else:
        feature_cols = [c for c in feature_cols if c in feat.columns]

    feat = feat.copy()
    feat.loc[:, feature_cols] = feat.loc[:, feature_cols].shift(1)
    return feat, feature_cols


def build_strategy_return_features(ret_use: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    rs = ret_use[["trade_date", "ret"]].copy().sort_values("trade_date").reset_index(drop=True)
    rs["ret_lag1"] = rs["ret"]
    for w in [3, 5, 10, 20, 40, 60]:
        rs[f"ret_r{w}"] = rs["ret"].rolling(w, min_periods=w).sum()
    for w in [5, 20, 60]:
        rs[f"ret_ewm{w}"] = rs["ret"].ewm(span=w, adjust=False).mean()
        rs[f"ret_vol{w}"] = rs["ret"].rolling(w, min_periods=w).std()
    for w in [20, 60]:
        rs[f"ret_skew{w}"] = rs["ret"].rolling(w, min_periods=w).skew()
    rs["ret_win5"] = rs["ret"].gt(0).rolling(5, min_periods=5).mean()
    rs["ret_win20"] = rs["ret"].gt(0).rolling(20, min_periods=20).mean()
    rs["ret_z20"] = rolling_zscore(rs["ret"], 20, 10)
    rs["ret_z60"] = rolling_zscore(rs["ret"], 60, 20)

    nav = (1 + rs["ret"].fillna(0)).cumprod()
    rs["ret_dd20"] = nav / nav.rolling(20, min_periods=5).max() - 1
    rs["ret_dd60"] = nav / nav.rolling(60, min_periods=20).max() - 1
    rs["ret_dd120"] = nav / nav.rolling(120, min_periods=40).max() - 1
    rs["ret_bad_streak5"] = rs["ret"].lt(0).rolling(5, min_periods=5).sum()
    rs["ret_bad_streak10"] = rs["ret"].lt(0).rolling(10, min_periods=10).sum()

    feature_cols = [c for c in rs.columns if c not in {"trade_date", "ret"}]
    rs.loc[:, feature_cols] = rs.loc[:, feature_cols].shift(1)
    return rs[["trade_date"] + feature_cols], feature_cols


def add_strategy_return_features(
    feat_lagged: pd.DataFrame,
    ret_use: pd.DataFrame,
) -> Tuple[pd.DataFrame, List[str]]:
    rs_feat, rs_cols = build_strategy_return_features(ret_use)
    feat = feat_lagged.merge(rs_feat, on="trade_date", how="left")

    interaction_specs = [
        ("risk_on_r5", "ret_r5", "risk_on_x_ret_r5"),
        ("risk_on_r20", "ret_r20", "risk_on_x_ret_r20"),
        ("style_spread_r5", "ret_r5", "style_x_ret_r5"),
        ("breadth_up_ratio_5", "ret_dd20", "breadth_x_dd20"),
        ("cross_dispersion_r1", "ret_vol20", "disp_x_ret_vol20"),
        ("main_vol_shock", "ret_dd20", "shock_x_dd20"),
        ("main_ma60_bias", "ret_dd60", "trend_x_dd60"),
        ("siphon_headwind", "ret_dd20", "headwind_x_dd20"),
        ("growth_lead_r20", "ret_r20", "growthlead_x_ret_r20"),
    ]
    interaction_cols: List[str] = []
    for left, right, out_col in interaction_specs:
        if left in feat.columns and right in feat.columns:
            feat[out_col] = feat[left] * feat[right]
            interaction_cols.append(out_col)

    return feat, rs_cols + interaction_cols


def summarize_feature_availability(
    feat_lagged: pd.DataFrame,
    feature_cols: List[str],
) -> Tuple[pd.DataFrame, pd.DataFrame, int]:
    rows = []
    for col in feature_cols:
        first_valid = feat_lagged.loc[feat_lagged[col].notna(), "trade_date"]
        if first_valid.empty:
            continue
        index_code = None
        if "__" in col:
            prefix = col.split("__", 1)[0]
            if prefix.isdigit() and len(prefix) == 6:
                index_code = prefix
        rows.append({
            "feature": col,
            "index_code": index_code,
            "first_valid_date": int(first_valid.iloc[0]),
        })

    feature_start = pd.DataFrame(rows).sort_values(["first_valid_date", "feature"]).reset_index(drop=True)
    if feature_start.empty:
        raise ValueError("No usable feature columns found")

    index_start = (
        feature_start[feature_start["index_code"].notna()]
        .groupby("index_code", as_index=False)
        .agg(
            first_usable_date=("first_valid_date", "max"),
            used_feature_count=("feature", "count"),
        )
        .sort_values(["first_usable_date", "index_code"])
        .reset_index(drop=True)
    )
    model_start_date = int(feature_start["first_valid_date"].max())
    return feature_start, index_start, model_start_date


def fit_label_spec(
    ret_series: pd.Series,
    method: str = "ternary_quantile",
    upper_q: float = 0.67,
    lower_q: float = 0.33,
    upper_thresh: float = 0.006,
    lower_thresh: float = -0.003,
) -> Dict[str, Any]:
    valid = ret_series.dropna()
    if len(valid) == 0:
        raise ValueError("No valid returns for label fitting")
    spec = {
        "method": method,
        "upper_q": upper_q,
        "lower_q": lower_q,
        "upper_thresh": upper_thresh,
        "lower_thresh": lower_thresh,
    }
    if method == "ternary_quantile":
        spec["q_lo"] = float(valid.quantile(lower_q))
        spec["q_hi"] = float(valid.quantile(upper_q))
    return spec


def apply_label_spec(ret_series: pd.Series, spec: Dict[str, Any]) -> pd.Series:
    s = ret_series.copy()
    labels = pd.Series(np.nan, index=s.index, dtype=float)
    method = spec["method"]

    if method == "ternary_quantile":
        labels.loc[s.notna()] = 1
        labels.loc[s >= spec["q_hi"]] = 2
        labels.loc[s <= spec["q_lo"]] = 0
    elif method == "ternary_threshold":
        labels.loc[s.notna()] = 1
        labels.loc[s > spec["upper_thresh"]] = 2
        labels.loc[s < spec["lower_thresh"]] = 0
    elif method == "binary_sign":
        labels.loc[s.notna()] = (s[s.notna()] > 0).astype(float)
    else:
        raise ValueError(f"Unknown method: {method}")

    return labels.astype("Int64")


def prepare_feature_block(
    feat: pd.DataFrame,
    feature_cols: List[str],
    fill_values: Optional[pd.Series] = None,
) -> Tuple[pd.DataFrame, pd.Series]:
    x = feat[["trade_date"] + feature_cols].copy().sort_values("trade_date").reset_index(drop=True)
    x.loc[:, feature_cols] = x.loc[:, feature_cols].ffill()
    if fill_values is None:
        fill_values = x.loc[:, feature_cols].median(numeric_only=True)
    x.loc[:, feature_cols] = x.loc[:, feature_cols].fillna(fill_values)
    x = x.dropna(subset=feature_cols).reset_index(drop=True)
    return x, fill_values


def build_classifier(clf_type: str = "lgbm", n_classes: int = 3):
    if clf_type == "lgbm_tuned" and _HAS_LGBM:
        return LGBMClassifier(
            n_estimators=300,
            learning_rate=0.02,
            num_leaves=7,
            max_depth=3,
            min_child_samples=60,
            subsample=0.9,
            colsample_bytree=0.6,
            reg_alpha=0.5,
            reg_lambda=2.0,
            class_weight="balanced",
            random_state=42,
            verbose=-1,
        )
    if clf_type == "lgbm_fast" and _HAS_LGBM:
        return LGBMClassifier(
            n_estimators=180,
            learning_rate=0.03,
            num_leaves=9,
            max_depth=3,
            min_child_samples=45,
            subsample=0.85,
            colsample_bytree=0.65,
            reg_alpha=0.3,
            reg_lambda=1.5,
            class_weight="balanced",
            random_state=42,
            verbose=-1,
        )
    if clf_type == "lgbm" and _HAS_LGBM:
        return LGBMClassifier(
            n_estimators=120,
            learning_rate=0.03,
            num_leaves=15,
            max_depth=3,
            min_child_samples=40,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_alpha=0.2,
            reg_lambda=1.0,
            class_weight="balanced",
            random_state=42,
            verbose=-1,
        )
    if clf_type == "logistic":
        return LogisticRegression(max_iter=1000, C=0.3, random_state=42)
    return GradientBoostingClassifier(
        n_estimators=120,
        learning_rate=0.03,
        max_depth=2,
        subsample=0.8,
        random_state=42,
    )


def fit_supervised_model(
    feat_lagged: pd.DataFrame,
    ret_use: pd.DataFrame,
    feature_cols: List[str],
    start_date: int,
    train_end: int = TRAIN_END,
    label_method: str = "ternary_quantile",
    upper_q: float = 0.67,
    lower_q: float = 0.33,
    upper_thresh: float = 0.006,
    lower_thresh: float = -0.003,
    clf_type: str = "lgbm",
) -> Dict[str, Any]:
    fit_feat = feat_lagged[
        (feat_lagged["trade_date"] >= start_date) & (feat_lagged["trade_date"] <= train_end)
    ].copy().reset_index(drop=True)
    fit_ret = ret_use[
        (ret_use["trade_date"] >= start_date) & (ret_use["trade_date"] <= train_end)
    ].copy().reset_index(drop=True)

    x_fit, fill_values = prepare_feature_block(fit_feat, feature_cols)
    x_fit = x_fit.merge(fit_ret, on="trade_date", how="inner")
    label_spec = fit_label_spec(
        x_fit["ret"],
        method=label_method,
        upper_q=upper_q,
        lower_q=lower_q,
        upper_thresh=upper_thresh,
        lower_thresh=lower_thresh,
    )
    y_fit = apply_label_spec(x_fit["ret"], label_spec)
    valid = y_fit.notna()
    if valid.sum() < 30 or y_fit[valid].nunique() < 2:
        raise ValueError("Not enough labeled rows to fit supervised model")

    scaler = StandardScaler()
    X_fit = scaler.fit_transform(x_fit.loc[valid, feature_cols].values)
    y = y_fit.loc[valid].astype(int).values

    clf = build_classifier(clf_type, len(np.unique(y)))
    clf.fit(X_fit, y)

    return {
        "model": clf,
        "scaler": scaler,
        "fill_values": fill_values,
        "label_spec": label_spec,
        "feature_cols": feature_cols,
        "train_frame": x_fit.loc[valid].copy().reset_index(drop=True),
        "y_train": y,
        "train_end": train_end,
        "last_label_date": int(ret_use["trade_date"].max()),
    }


def predict_full_sample(
    feat_lagged: pd.DataFrame,
    ret_use: pd.DataFrame,
    fit_pack: Dict[str, Any],
    start_date: int,
) -> pd.DataFrame:
    feature_cols = fit_pack["feature_cols"]
    extra_cols = [
        c for c in [
            "headwind_score_raw",
            "growth_lead_r20",
            "growth_strength_gap",
            "growth_breadth_gap",
            "siphon_headwind",
            "rotation_stress",
        ] if c in feat_lagged.columns
    ]
    full_feat = feat_lagged[feat_lagged["trade_date"] >= start_date].copy().reset_index(drop=True)
    x_full, _ = prepare_feature_block(full_feat, feature_cols, fill_values=fit_pack["fill_values"])

    X_full = fit_pack["scaler"].transform(x_full[feature_cols].values)
    pred = fit_pack["model"].predict(X_full)
    out = x_full[["trade_date"] + extra_cols].copy()
    out["state"] = pred.astype(int)
    out["state_name"] = out["state"].map({0: "bad", 1: "neutral", 2: "good"}).fillna("unknown")

    if hasattr(fit_pack["model"], "predict_proba"):
        prob = fit_pack["model"].predict_proba(X_full)
        for i in range(prob.shape[1]):
            out[f"prob_{i}"] = prob[:, i]

    out = out.merge(ret_use, on="trade_date", how="left")
    out["ret_label"] = apply_label_spec(out["ret"], fit_pack["label_spec"])
    out["has_label"] = out["ret"].notna()
    out["is_after_train_end"] = out["trade_date"] > fit_pack["train_end"]
    out["is_after_last_label_date"] = out["trade_date"] > fit_pack["last_label_date"]
    return out.sort_values("trade_date").reset_index(drop=True)


def feature_importance_table(fit_pack: Dict[str, Any]) -> pd.DataFrame:
    clf = fit_pack["model"]
    feature_cols = fit_pack["feature_cols"]
    if hasattr(clf, "feature_importances_"):
        imp = clf.feature_importances_
    elif hasattr(clf, "coef_"):
        coef = np.asarray(clf.coef_)
        imp = np.abs(coef).mean(axis=0)
    else:
        return pd.DataFrame()
    return pd.DataFrame({
        "feature": feature_cols,
        "importance": imp,
    }).sort_values("importance", ascending=False).reset_index(drop=True)


def score_validation_result(validation_df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, float]]:
    perf = summarize_strategy_by_state(validation_df, state_col="state", ret_col="ret")
    if len(perf) == 0:
        return perf, {
            "n_states": 0,
            "min_state_frac": np.nan,
            "best_mean_ret": np.nan,
            "worst_mean_ret": np.nan,
            "target_hit": 0.0,
            "target_gap": np.inf,
        }
    worst_mean = float(perf["mean_ret"].min())
    return perf, {
        "n_states": len(perf),
        "min_state_frac": float(perf["count_pct"].min()),
        "best_mean_ret": float(perf["mean_ret"].max()),
        "worst_mean_ret": worst_mean,
        "target_hit": float((perf["mean_ret"] <= -0.001).any()),
        "target_gap": abs(worst_mean + 0.001),
    }


def tune_holdout_configs(
    feat_lagged: pd.DataFrame,
    ret_use: pd.DataFrame,
    feature_cols: List[str],
    start_date: int,
    candidate_configs: List[Dict[str, Any]],
    holdout_start: int = HOLDOUT_START,
    holdout_end: int = HOLDOUT_END,
) -> pd.DataFrame:
    rows = []
    for cfg in candidate_configs:
        fit_pack = fit_supervised_model(
            feat_lagged=feat_lagged,
            ret_use=ret_use,
            feature_cols=feature_cols,
            start_date=start_date,
            train_end=TRAIN_END,
            **cfg,
        )
        full_pred = predict_full_sample(
            feat_lagged=feat_lagged,
            ret_use=ret_use,
            fit_pack=fit_pack,
            start_date=start_date,
        )
        validation_oos = full_pred[
            (full_pred["trade_date"] >= holdout_start)
            & (full_pred["trade_date"] <= holdout_end)
            & full_pred["has_label"]
        ].copy().reset_index(drop=True)
        perf, score = score_validation_result(validation_oos)
        rows.append({
            **cfg,
            "validation_rows": len(validation_oos),
            **score,
            "bad_state_share": float(perf.loc[perf["mean_ret"] == perf["mean_ret"].min(), "count_pct"].iloc[0]) if len(perf) else np.nan,
        })

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["target_hit", "target_gap", "min_state_frac", "best_mean_ret"],
            ascending=[False, True, False, False],
        )
        .reset_index(drop=True)
    )


def summarize_recent_headwinds(
    feat_lagged: pd.DataFrame,
    start_date: int = RECENT_PRIOR_START,
    end_date: int = RECENT_PRIOR_END,
    baseline_end: int = HOLDOUT_END,
) -> pd.DataFrame:
    factor_cols = [
        "growth_lead_r20",
        "growth_strength_gap",
        "growth_breadth_gap",
        "siphon_headwind",
        "rotation_stress",
        "headwind_score_raw",
        "style_spread_r20",
        "main_r20",
        "growth_r20",
    ]
    available = [c for c in factor_cols if c in feat_lagged.columns]
    hist = feat_lagged.loc[feat_lagged["trade_date"] <= baseline_end, ["trade_date"] + available].copy()
    recent = feat_lagged.loc[
        (feat_lagged["trade_date"] >= start_date) & (feat_lagged["trade_date"] <= end_date),
        ["trade_date"] + available,
    ].copy()
    rows = []
    for col in available:
        hist_s = hist[col].dropna()
        recent_s = recent[col].dropna()
        if hist_s.empty or recent_s.empty:
            continue
        recent_mean = float(recent_s.mean())
        rows.append({
            "factor": col,
            "recent_mean": recent_mean,
            "hist_pctile": float((hist_s < recent_mean).mean()),
            "hist_q05": float(hist_s.quantile(0.05)),
            "hist_q95": float(hist_s.quantile(0.95)),
        })
    return pd.DataFrame(rows).sort_values("hist_pctile", ascending=False).reset_index(drop=True)


## 5. Load Data

In [ ]:

df_raw = pd.read_csv(INDEX_PATH)
df_raw["trade_date"] = df_raw["trade_date"].astype(int)

df_ret = pd.read_csv(RETURN_PATH)
df_ret["date"] = df_ret["date"].astype(int)

print(f"index rows = {len(df_raw):,}, date range = {df_raw['trade_date'].min()} ~ {df_raw['trade_date'].max()}")
print(f"return rows = {len(df_ret):,}, date range = {df_ret['date'].min()} ~ {df_ret['date'].max()}")


## 6. Build Feature Matrix & Resolve Usable Start Date


In [ ]:
df_ret_use = build_return_table(df_ret)
feat_base, base_feature_cols = build_feature_matrix(df_raw)
base_feature_start_info, index_start_info, base_model_start_date = summarize_feature_availability(feat_base, base_feature_cols)

baseline_fit = fit_supervised_model(
    feat_lagged=feat_base,
    ret_use=df_ret_use,
    feature_cols=base_feature_cols,
    start_date=base_model_start_date,
    train_end=TRAIN_END,
    label_method="ternary_quantile",
    upper_q=0.67,
    lower_q=0.33,
    clf_type="lgbm",
)
base_top_features = feature_importance_table(baseline_fit).head(30)["feature"].tolist()

feat_lagged, strategy_feature_cols = add_strategy_return_features(feat_base, df_ret_use)
headwind_feature_cols = [
    c for c in [
        "growth_lead_r5", "growth_lead_r20", "growth_lead_r60",
        "growth_strength_gap", "growth_breadth_gap",
        "siphon_headwind", "rotation_stress", "growth_vol_headwind", "headwind_score_raw",
        "growth_lead_r5_chg5", "growth_lead_r20_chg5", "growth_strength_gap_chg5",
        "siphon_headwind_chg5", "rotation_stress_chg5", "headwind_score_raw_chg5",
    ] if c in feat_lagged.columns
]
feature_cols = list(dict.fromkeys([c for c in base_top_features + strategy_feature_cols + headwind_feature_cols if c in feat_lagged.columns]))
feature_start_info, _, model_start_date = summarize_feature_availability(feat_lagged, feature_cols)

print(f"feature matrix rows = {len(feat_lagged):,}")
print(f"base feature count = {len(base_feature_cols)}")
print(f"selected feature count = {len(feature_cols)}")
print(f"train_end = {TRAIN_END}")
print(f"validation window = {HOLDOUT_START} ~ {HOLDOUT_END}")
print(f"model_start_date = {model_start_date}")
print("top base features:", base_top_features)
print("strategy return features:", strategy_feature_cols)
print("explicit headwind features:", headwind_feature_cols)

display(index_start_info)


## 7. 2025 Full-Year Validation Search


In [ ]:
candidate_configs = [
    {"label_method": "ternary_threshold", "clf_type": "lgbm", "upper_thresh": 0.008, "lower_thresh": -0.004},
    {"label_method": "ternary_threshold", "clf_type": "lgbm", "upper_thresh": 0.009, "lower_thresh": -0.004},
    {"label_method": "ternary_threshold", "clf_type": "lgbm", "upper_thresh": 0.009, "lower_thresh": -0.005},
    {"label_method": "ternary_threshold", "clf_type": "lgbm_fast", "upper_thresh": 0.008, "lower_thresh": -0.005},
    {"label_method": "ternary_threshold", "clf_type": "lgbm_fast", "upper_thresh": 0.009, "lower_thresh": -0.003},
    {"label_method": "ternary_threshold", "clf_type": "lgbm_tuned", "upper_thresh": 0.007, "lower_thresh": -0.003},
    {"label_method": "ternary_threshold", "clf_type": "lgbm_tuned", "upper_thresh": 0.009, "lower_thresh": -0.005},
    {"label_method": "ternary_quantile", "clf_type": "lgbm", "upper_q": 0.70, "lower_q": 0.30},
    {"label_method": "ternary_quantile", "clf_type": "lgbm_tuned", "upper_q": 0.70, "lower_q": 0.30},
    {"label_method": "ternary_quantile", "clf_type": "lgbm_fast", "upper_q": 0.70, "lower_q": 0.30},
    {"label_method": "ternary_threshold", "clf_type": "logistic", "upper_thresh": 0.008, "lower_thresh": -0.005},
]

config_search = tune_holdout_configs(
    feat_lagged=feat_lagged,
    ret_use=df_ret_use,
    feature_cols=feature_cols,
    start_date=model_start_date,
    candidate_configs=candidate_configs,
    holdout_start=HOLDOUT_START,
    holdout_end=HOLDOUT_END,
)

best_cfg = config_search.iloc[0].to_dict()
print("selected best config:")
print(best_cfg)
display(config_search.round(6))


In [ ]:
# fixed configuration is set in the previous cell


## 8. 2025 Full-Year Pure Out-of-Sample Evaluation


In [ ]:
fit_pack = fit_supervised_model(
    feat_lagged=feat_lagged,
    ret_use=df_ret_use,
    feature_cols=feature_cols,
    start_date=model_start_date,
    train_end=TRAIN_END,
    label_method=best_cfg["label_method"],
    upper_q=best_cfg.get("upper_q", 0.67),
    lower_q=best_cfg.get("lower_q", 0.33),
    upper_thresh=best_cfg.get("upper_thresh", 0.006),
    lower_thresh=best_cfg.get("lower_thresh", -0.003),
    clf_type=best_cfg["clf_type"],
)
full_pred = predict_full_sample(
    feat_lagged=feat_lagged,
    ret_use=df_ret_use,
    fit_pack=fit_pack,
    start_date=model_start_date,
)

validation_oos = full_pred[
    (full_pred["trade_date"] >= HOLDOUT_START)
    & (full_pred["trade_date"] <= HOLDOUT_END)
    & (full_pred["has_label"])
].copy().reset_index(drop=True)
validation_perf, validation_score = score_validation_result(validation_oos)

print(f"training rows = {len(fit_pack['train_frame']):,}")
print(f"training window = {int(fit_pack['train_frame']['trade_date'].min())} ~ {int(fit_pack['train_frame']['trade_date'].max())}")
print(f"validation rows = {len(validation_oos):,}")
print("validation score:", validation_score)
display(validation_perf.round(6))
oos_eval = evaluate_oos_states(validation_oos, label="2025 Full-Year Pure OOS")


In [ ]:

oos_with_labels = validation_oos.copy()
cm_df = oos_with_labels[["state", "ret_label"]].dropna().copy()
if len(cm_df):
    y_true = cm_df["ret_label"].astype(int)
    y_pred = cm_df["state"].astype(int)
    used_labels = sorted(set(y_true) | set(y_pred))
    print("confusion matrix")
    display(pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=used_labels),
        index=[f"true_{x}" for x in used_labels],
        columns=[f"pred_{x}" for x in used_labels],
    ))
    print(classification_report(y_true, y_pred, zero_division=0))


## 9. Training Sample Diagnostics


In [ ]:
print("model fitted on in-sample window only")
print("train_end:", fit_pack["train_end"])
print("validation window:", HOLDOUT_START, "~", HOLDOUT_END)
print("last labeled date:", fit_pack["last_label_date"])

recent_headwinds = summarize_recent_headwinds(feat_lagged)
print(f"recent prior window: {RECENT_PRIOR_START} ~ {RECENT_PRIOR_END}")
if len(recent_headwinds):
    print("recent adverse factors ranked by historical percentile")
    display(recent_headwinds.round(6))


In [ ]:

fi = feature_importance_table(fit_pack)
fi.head(25)


## 10. Full-Sample Prediction, Recent Prior Diagnostics & Export


In [ ]:
recent_pred = full_pred[
    (full_pred["trade_date"] >= RECENT_PRIOR_START)
    & (full_pred["trade_date"] <= RECENT_PRIOR_END)
].copy().reset_index(drop=True)

full_pred.to_csv(OUTPUT_PATH, index=False)
print(f"saved to: {OUTPUT_PATH}")
print(f"prediction rows = {len(full_pred):,}")
print(f"labeled rows = {int(full_pred['has_label'].sum()):,}")
print(f"2025 full-year labeled OOS rows = {int(((full_pred['trade_date'] >= HOLDOUT_START) & (full_pred['trade_date'] <= HOLDOUT_END) & full_pred['has_label']).sum()):,}")
print(f"2026 labeled rows = {int(((full_pred['trade_date'] >= 20260101) & full_pred['has_label']).sum()):,}")
print(f"recent prior rows = {len(recent_pred):,}")
print(f"unlabeled future rows = {int((~full_pred['has_label']).sum()):,}")
if len(recent_pred):
    print("recent prior state counts")
    print(recent_pred["state"].value_counts().sort_index())
    diagnostic_cols = [
        c for c in [
            "trade_date", "state", "state_name", "headwind_score_raw",
            "growth_lead_r20", "growth_strength_gap", "growth_breadth_gap",
            "siphon_headwind", "rotation_stress",
        ] if c in recent_pred.columns
    ]
    display(recent_pred[diagnostic_cols].head(20).round(6))


In [ ]:
# full_pred[['trade_date', 'state']].to_csv(OUTPUT_PATH, index=False)